> ## SOLUTION / ANSWER KEY &mdash; Lab 9.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-validate-grounding.ipynb`](../lab-07-validate-grounding.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.7 &mdash; Validate Grounding

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Check every claim maps to a real figure in the report
- Check each claim cites the correct source
- Refuse to ship a summary with an ungrounded claim

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it** (the real grounding / citation / compute logic, or the real `create_agent` wiring), then **Run it** and read the output &mdash; and, for the agent labs, the real **message trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working, grounded insight agent you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The insight agent is a **real** `create_agent` driven by a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")`, key in `.env` as `GROQ_API_KEY`). You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It is **informational only**: it grounds &amp; **cites** every figure, gives **no advice**, and has **no trade tool** (`place_trade` is defined but never bound &mdash; a human analyst decides). A `@tool` must **catch its own errors and return a string** &mdash; a tool that raises can abort the whole run. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing.

**Reference:** [Module 9 slides &mdash; Grounding: RAG & citations](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (the Day-5 provider)

WORK = "/tmp/biaa-lab-09-07"
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is set. Day-5 labs call a REAL hosted model (Groq)."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model. openai/gpt-oss-20b is a reliable tool-caller via create_agent.
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0) if groq_ready() else None

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("GROQ_API_KEY set | model: openai/gpt-oss-20b | WORK:", WORK)
else:
    print("GROQ_API_KEY NOT set -- add it to .env (free at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Never ship an ungrounded claim (deck slides 4, 8, 14). Before the summary goes to an analyst, the agent
**validates** it: every claim must map to a **real figure** in the report, and it must cite the **correct
source**. A claim that cites the wrong page, or a figure that isn't in the report, is a grounding failure
&mdash; don't ship it. This is the gate that turns "cited" (Lab 2) into "**correctly** cited".

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("a claim must match REPORT[metric] on the source string, e.g. revenue -> 'p4, income stmt'")

## Build it
Complete `validate_summary`: collect an ungrounded claim and a wrong-source claim, and return `ok` only
when there are no problems. Then run the cell.

In [ ]:
# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

def validate_summary(claims, report):
    problems = []
    for c in claims:
        fig = report.get(c['metric'])
        if fig is None:
            problems.append("ungrounded: " + c["metric"])   # cites a figure not in the report
        elif c["source"] != fig["source"]:
            problems.append("wrong source: " + c["metric"])
    return {"ok": len(problems) == 0, "problems": problems}

try:
    good = [{"metric": "revenue", "source": "p4, income stmt"}]
    ungrounded = [{"metric": "ebitda", "source": "p9"}]
    wrong = [{"metric": "revenue", "source": "p1, cover"}]
    print("good      ->", validate_summary(good, REPORT))
    print("ungrounded->", validate_summary(ungrounded, REPORT))
    print("wrong src ->", validate_summary(wrong, REPORT))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- A claim about a figure **not in the report** is `ungrounded`; a claim citing the **wrong page** is `wrong source`.
- `ok` is True only when the `problems` list is empty &mdash; one bad claim blocks the whole summary.
- This is the validation Lab 12's capstone pipeline runs before it flags a report `needs_review`.

## Your turn (open task &mdash; no grader)
Feed `validate_summary` a mixed list with one good claim, one ungrounded, and one wrong-source, and read
the collected problems. **What good looks like:** every problem is named individually, and `ok` is False, so
nothing ungrounded or mis-cited can slip through to an analyst.

---
### Key takeaway
Validation is the gate before an analyst sees the summary: every claim must map to a real figure and cite the right source. An ungrounded or mis-cited claim doesn't ship -- that's the high-stakes bar.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>